In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from tensorflow import keras
from tensorflow.keras import layers

import joblib

In [2]:
df = pd.read_csv("Netflix_Customer_churn.csv")

print(df.head())
print(df.info())

                            customer_id  age  gender subscription_type  \
0  a9b75100-82a8-427a-a208-72f24052884a   51   Other             Basic   
1  49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b   47   Other          Standard   
2  4d71f6ce-fca9-4ff7-8afa-197ac24de14b   27  Female          Standard   
3  d3c72c38-631b-4f9e-8a0e-de103cad1a7d   53   Other           Premium   
4  4e265c34-103a-4dbb-9553-76c9aa47e946   56   Other          Standard   

   watch_hours  last_login_days   region  device  monthly_fee  churned  \
0        14.73               29   Africa      TV         8.99        1   
1         0.70               19   Europe  Mobile        13.99        1   
2        16.32               10     Asia      TV        13.99        0   
3         4.51               12  Oceania      TV        17.99        1   
4         1.89               13   Africa  Mobile        13.99        1   

  payment_method  number_of_profiles  avg_watch_time_per_day favorite_genre  
0      Gift Card                

In [3]:
# drop useless column
df = df.drop("customer_id", axis=1)

# clean column names
df.columns = df.columns.str.strip()

print(df.columns)

Index(['age', 'gender', 'subscription_type', 'watch_hours', 'last_login_days',
       'region', 'device', 'monthly_fee', 'churned', 'payment_method',
       'number_of_profiles', 'avg_watch_time_per_day', 'favorite_genre'],
      dtype='object')


In [4]:
df = pd.get_dummies(df, drop_first=True)

print(df.head())
print(df.dtypes)

   age  watch_hours  last_login_days  monthly_fee  churned  \
0   51        14.73               29         8.99        1   
1   47         0.70               19        13.99        1   
2   27        16.32               10        13.99        0   
3   53         4.51               12        17.99        1   
4   56         1.89               13        13.99        1   

   number_of_profiles  avg_watch_time_per_day  gender_Male  gender_Other  \
0                   1                    0.49        False          True   
1                   5                    0.03        False          True   
2                   2                    1.48        False         False   
3                   2                    0.35        False          True   
4                   2                    0.13        False          True   

   subscription_type_Premium  ...  payment_method_Crypto  \
0                      False  ...                  False   
1                      False  ...                 

In [47]:
categorical_cols = ["subscription_type", "region", "device", "payment_method", "favorite_genre"]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# verify
print(df.columns)

Index(['age', 'gender', 'watch_hours', 'last_login_days', 'monthly_fee',
       'churned', 'number_of_profiles', 'avg_watch_time_per_day',
       'subscription_type_Premium', 'subscription_type_Standard',
       'region_Asia', 'region_Europe', 'region_North America',
       'region_Oceania', 'region_South America', 'device_Laptop',
       'device_Mobile', 'device_TV', 'device_Tablet', 'payment_method_Crypto',
       'payment_method_Debit Card', 'payment_method_Gift Card',
       'payment_method_PayPal', 'favorite_genre_Comedy',
       'favorite_genre_Documentary', 'favorite_genre_Drama',
       'favorite_genre_Horror', 'favorite_genre_Romance',
       'favorite_genre_Sci-Fi'],
      dtype='object')


In [5]:
df = pd.get_dummies(df, columns=["gender"], drop_first=True)

print(df.columns)

KeyError: "None of [Index(['gender'], dtype='object')] are in the [columns]"

In [49]:
# convert all boolean columns to int
bool_cols = X.select_dtypes(include=["bool"]).columns

X[bool_cols] = X[bool_cols].astype(int)

# verify
print(X.dtypes)

age                             int64
watch_hours                   float64
last_login_days                 int64
monthly_fee                   float64
number_of_profiles              int64
avg_watch_time_per_day        float64
subscription_type_Premium       int64
subscription_type_Standard      int64
region_Asia                     int64
region_Europe                   int64
region_North America            int64
region_Oceania                  int64
region_South America            int64
device_Laptop                   int64
device_Mobile                   int64
device_TV                       int64
device_Tablet                   int64
payment_method_Crypto           int64
payment_method_Debit Card       int64
payment_method_Gift Card        int64
payment_method_PayPal           int64
favorite_genre_Comedy           int64
favorite_genre_Documentary      int64
favorite_genre_Drama            int64
favorite_genre_Horror           int64
favorite_genre_Romance          int64
favorite_gen

In [6]:
y = df["churned"]
X = df.drop("churned", axis=1)

print(X.shape, y.shape)

(5000, 29) (5000,)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)


(4000, 29) (1000, 29)


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train[:2])

[[ 0.64767647 -0.15948307  1.48011461  1.16622218  0.69478561 -0.26447567
  -0.69824196  1.42811543  1.39403755 -0.69191262 -0.44855457 -0.45337127
   2.18831259 -0.42419453 -0.46494637 -0.49921863 -0.50896898 -0.49491658
   1.94542772  2.05639446 -0.51519155 -0.50039059 -0.50468321 -0.39801127
  -0.40931934 -0.41222898 -0.40890296  2.39932423 -0.41264393]
 [-0.83830161 -0.11177607  0.68172496 -1.2614405  -1.42669717 -0.23857389
  -0.69824196  1.42811543 -0.71734079 -0.69191262 -0.44855457 -0.45337127
   2.18831259 -0.42419453 -0.46494637 -0.49921863 -0.50896898 -0.49491658
   1.94542772 -0.48628803  1.94102562 -0.50039059 -0.50468321 -0.39801127
  -0.40931934 -0.41222898 -0.40890296 -0.41678402 -0.41264393]]


In [12]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.4),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
100/100 [==============================] - 0s 2ms/step - loss: 0.5805 - accuracy: 0.6844 - val_loss: 0.3463 - val_accuracy: 0.8550
Epoch 2/50
100/100 [==============================] - 0s 787us/step - loss: 0.3768 - accuracy: 0.8294 - val_loss: 0.2766 - val_accuracy: 0.8825
Epoch 3/50
100/100 [==============================] - 0s 792us/step - loss: 0.3048 - accuracy: 0.8631 - val_loss: 0.2681 - val_accuracy: 0.8875
Epoch 4/50
100/100 [==============================] - 0s 785us/step - loss: 0.2807 - accuracy: 0.8744 - val_loss: 0.2611 - val_accuracy: 0.8838
Epoch 5/50
100/100 [==============================] - 0s 760us/step - loss: 0.2583 - accuracy: 0.8853 - val_loss: 0.2600 - val_accuracy: 0.8925
Epoch 6/50
100/100 [==============================] - 0s 779us/step - loss: 0.2431 - accuracy: 0.8950 - val_loss: 0.2668 - val_accuracy: 0.8913
Epoch 7/50
100/100 [==============================] - 0s 768us/step - loss: 0.2267 - accuracy: 0.9047 - val_loss: 0.2623 - val_accuracy: 0

In [14]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

32/32 [==============================] - 0s 448us/step - loss: 0.2630 - accuracy: 0.8940
Test Loss: 0.2630009651184082
Test Accuracy: 0.8939999938011169


In [15]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

32/32 [==============================] - 0s 365us/step
[[445  53]
 [ 53 449]]
              precision    recall  f1-score   support

           0       0.89      0.89      0.89       498
           1       0.89      0.89      0.89       502

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000



In [16]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[445  53]
 [ 53 449]]
              precision    recall  f1-score   support

           0       0.89      0.89      0.89       498
           1       0.89      0.89      0.89       502

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000



In [26]:
sample = pd.DataFrame([{
    "age": 28,
    "gender": "Female",
    "subscription_type": "Premium",
    "watch_hours": 25,
    "last_login_days": 1,
    "region": "Europe",
    "device": "TV",
    "monthly_fee": 17.99,
    "payment_method": "Debit Card",
    "number_of_profiles": 4,
    "avg_watch_time_per_day": 3.5,
    "favorite_genre": "Drama"
}])

In [27]:
# same preprocessing as training
sample = pd.get_dummies(sample)

# align columns with training data
sample = sample.reindex(columns=X.columns, fill_value=0)

In [28]:
sample_scaled = scaler.transform(sample)

In [29]:
prob = model.predict(sample_scaled)
print("Churn Probability:", prob[0][0])
print("Prediction:", prob[0][0] > 0.5)

1/1 [==============================] - 0s 41ms/step
Churn Probability: 9.452025e-19
Prediction: False


In [30]:
import joblib

model.save("model.h5")
joblib.dump(scaler, "scaler.pkl")

/Users/apple/Desktop/mission D/customer_churn/venv/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


['scaler.pkl']

In [31]:
for days in [1, 5, 10, 20, 40]:
    sample["last_login_days"] = days
    sample_enc = pd.get_dummies(sample)
    sample_enc = sample_enc.reindex(columns=X.columns, fill_value=0)
    sample_scaled = scaler.transform(sample_enc)

    prob = model.predict(sample_scaled)[0][0]
    print(days, "->", prob)

1/1 [==============================] - 0s 10ms/step
1 -> 9.452025e-19
1/1 [==============================] - 0s 9ms/step
5 -> 5.0219847e-18
1/1 [==============================] - 0s 8ms/step
10 -> 4.200127e-17
1/1 [==============================] - 0s 8ms/step
20 -> 2.3778839e-15
1/1 [==============================] - 0s 7ms/step
40 -> 7.0314657e-12


In [32]:
cases = [
    {"watch_hours": 1, "last_login_days": 40},
    {"watch_hours": 0.2, "last_login_days": 50},
    {"watch_hours": 25, "last_login_days": 2},
]